# Synthetic Data Generation Framework for Privacy-Preserving AI

**MTech Project Demo Notebook**  
This notebook walks through the complete end-to-end pipeline for generating realistic synthetic patient data using generative AI models, with built-in differential privacy and bias auditing — without using any real patient data.

In [ ]:
# Setup: add project root to path
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # use non-interactive backend; remove for inline display
import seaborn as sns

# Use inline display for Jupyter
%matplotlib inline
sns.set_style('whitegrid')

import config
print('✓ Environment ready')

---
## Step 1: Load Real Data (Synthea)

In [ ]:
from modules.data_loader import load_synthea_data

train_df, test_df, full_df = load_synthea_data()

print(f'\nFull dataset shape : {full_df.shape}')
print(f'Train split shape  : {train_df.shape}')
print(f'Test  split shape  : {test_df.shape}')

In [ ]:
# Preview the first 5 rows
full_df.head()

In [ ]:
# Column-level summary
full_df.describe(include='all').T

---
## Step 2: Train Generative Model

In [ ]:
from modules.gan_trainer import train_all_models

# Set force_retrain=True to always retrain; False = load from disk if available
models = train_all_models(train_df, force_retrain=False)
print('\nTrained models:', list(models.keys()))

In [ ]:
# Display training loss curve inline
from IPython.display import Image, display

loss_chart = config.TRAINING_LOSS_PNG
if os.path.exists(loss_chart):
    display(Image(loss_chart, width=700))
else:
    print('Training loss chart not found — run main.py first or retrain above.')

---
## Step 3: Generate Synthetic Data

In [ ]:
from modules.gan_trainer import generate_synthetic

ctgan_model = models['ctgan']
synthetic_df = generate_synthetic(ctgan_model, n=config.N_SYNTHETIC_SAMPLES)

print(f'\nSynthetic data shape: {synthetic_df.shape}')

In [ ]:
# Preview synthetic data
synthetic_df.head()

In [ ]:
# Quick data-type comparison
print('=== Real Data dtypes ===')
print(full_df.dtypes)
print('\n=== Synthetic Data dtypes ===')
print(synthetic_df.dtypes)

---
## Step 4: Side-by-Side Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
%matplotlib inline

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
sns.set_style('whitegrid')

# ── AGE distribution ─────────────────────────────────────────────────────────
if 'AGE' in full_df.columns and 'AGE' in synthetic_df.columns:
    sns.histplot(full_df['AGE'], ax=axes[0, 0], color='#4C72B0',
                 kde=True, label='Real', stat='density', bins=30)
    sns.histplot(synthetic_df['AGE'], ax=axes[0, 0], color='#DD8452',
                 kde=True, label='Synthetic', alpha=0.6, stat='density', bins=30)
    axes[0, 0].set_title('AGE Distribution', fontweight='bold')
    axes[0, 0].legend()

# ── GENDER distribution ───────────────────────────────────────────────────────
if 'GENDER' in full_df.columns and 'GENDER' in synthetic_df.columns:
    real_gender = full_df['GENDER'].astype(str).value_counts(normalize=True)
    syn_gender  = synthetic_df['GENDER'].astype(str).value_counts(normalize=True)
    all_cats    = sorted(set(real_gender.index) | set(syn_gender.index))
    x = range(len(all_cats))
    axes[0, 1].bar([i - 0.2 for i in x],
                   [real_gender.get(c, 0) for c in all_cats],
                   width=0.4, label='Real', color='#4C72B0')
    axes[0, 1].bar([i + 0.2 for i in x],
                   [syn_gender.get(c, 0) for c in all_cats],
                   width=0.4, label='Synthetic', color='#DD8452', alpha=0.8)
    axes[0, 1].set_xticks(list(x))
    axes[0, 1].set_xticklabels(all_cats)
    axes[0, 1].set_title('GENDER Distribution', fontweight='bold')
    axes[0, 1].legend()

# ── RACE distribution ────────────────────────────────────────────────────────
if 'RACE' in full_df.columns and 'RACE' in synthetic_df.columns:
    real_race = full_df['RACE'].astype(str).value_counts(normalize=True)
    syn_race  = synthetic_df['RACE'].astype(str).value_counts(normalize=True)
    all_cats  = sorted(set(real_race.index) | set(syn_race.index))
    x = range(len(all_cats))
    axes[0, 2].bar([i - 0.2 for i in x],
                   [real_race.get(c, 0) for c in all_cats],
                   width=0.4, label='Real', color='#4C72B0')
    axes[0, 2].bar([i + 0.2 for i in x],
                   [syn_race.get(c, 0) for c in all_cats],
                   width=0.4, label='Synthetic', color='#DD8452', alpha=0.8)
    axes[0, 2].set_xticks(list(x))
    axes[0, 2].set_xticklabels(all_cats, rotation=35, ha='right', fontsize=8)
    axes[0, 2].set_title('RACE Distribution', fontweight='bold')
    axes[0, 2].legend()

# ── HEALTHCARE_EXPENSES distribution ─────────────────────────────────────────
if 'HEALTHCARE_EXPENSES' in full_df.columns:
    sns.histplot(full_df['HEALTHCARE_EXPENSES'], ax=axes[1, 0],
                 color='#4C72B0', kde=True, label='Real', stat='density', bins=30)
    sns.histplot(synthetic_df.get('HEALTHCARE_EXPENSES',
                                  pd.Series(dtype=float)),
                 ax=axes[1, 0], color='#DD8452', kde=True,
                 label='Synthetic', alpha=0.6, stat='density', bins=30)
    axes[1, 0].set_title('HEALTHCARE_EXPENSES', fontweight='bold')
    axes[1, 0].legend()

# ── MARITAL distribution ─────────────────────────────────────────────────────
if 'MARITAL' in full_df.columns:
    real_m = full_df['MARITAL'].astype(str).value_counts(normalize=True)
    syn_m  = synthetic_df['MARITAL'].astype(str).value_counts(normalize=True)
    all_c  = sorted(set(real_m.index) | set(syn_m.index))
    x = range(len(all_c))
    axes[1, 1].bar([i - 0.2 for i in x],
                   [real_m.get(c, 0) for c in all_c],
                   width=0.4, label='Real', color='#4C72B0')
    axes[1, 1].bar([i + 0.2 for i in x],
                   [syn_m.get(c, 0) for c in all_c],
                   width=0.4, label='Synthetic', color='#DD8452', alpha=0.8)
    axes[1, 1].set_xticks(list(x))
    axes[1, 1].set_xticklabels(all_c)
    axes[1, 1].set_title('MARITAL Distribution', fontweight='bold')
    axes[1, 1].legend()

# ── ETHNICITY distribution ────────────────────────────────────────────────────
if 'ETHNICITY' in full_df.columns:
    real_e = full_df['ETHNICITY'].astype(str).value_counts(normalize=True)
    syn_e  = synthetic_df['ETHNICITY'].astype(str).value_counts(normalize=True)
    all_c  = sorted(set(real_e.index) | set(syn_e.index))
    x = range(len(all_c))
    axes[1, 2].bar([i - 0.2 for i in x],
                   [real_e.get(c, 0) for c in all_c],
                   width=0.4, label='Real', color='#4C72B0')
    axes[1, 2].bar([i + 0.2 for i in x],
                   [syn_e.get(c, 0) for c in all_c],
                   width=0.4, label='Synthetic', color='#DD8452', alpha=0.8)
    axes[1, 2].set_xticks(list(x))
    axes[1, 2].set_xticklabels(all_c, rotation=20, ha='right')
    axes[1, 2].set_title('ETHNICITY Distribution', fontweight='bold')
    axes[1, 2].legend()

fig.suptitle('Real vs. Synthetic Data: Distribution Comparison',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()

dist_path = config.DISTRIBUTION_PLOT_PNG
os.makedirs(os.path.dirname(dist_path), exist_ok=True)
fig.savefig(dist_path, dpi=config.CHART_DPI, bbox_inches='tight')
print(f'Distribution chart saved to {dist_path}')
plt.show()

---
## Step 5: Privacy Evaluation

In [ ]:
from modules.privacy_layer import apply_differential_privacy, run_membership_inference_attack

print(f'Differential Privacy Budget (ε) : {config.DP_EPSILON}')
print(f'Mechanism                        : Laplace')
print(f'Protected columns                : {config.NUMERIC_COLUMNS}')
print(f'Sensitivity                      : {config.DP_SENSITIVITY}')
print()

# Apply DP to synthetic data
dp_synthetic_df = apply_differential_privacy(synthetic_df, epsilon=config.DP_EPSILON)

# Compare AGE before/after DP
if 'AGE' in dp_synthetic_df.columns:
    print(f'AGE mean  — before DP: {synthetic_df["AGE"].mean():.2f}')
    print(f'AGE mean  — after  DP: {dp_synthetic_df["AGE"].mean():.2f}')
    print(f'AGE stdev — before DP: {synthetic_df["AGE"].std():.2f}')
    print(f'AGE stdev — after  DP: {dp_synthetic_df["AGE"].std():.2f}')

In [ ]:
# Membership inference attack evaluation
privacy_risk_score = run_membership_inference_attack(full_df, dp_synthetic_df)
print(f'\nPrivacy Risk Score: {privacy_risk_score}/100 (lower = more private)')

---
## Step 6: Quality Metrics

In [ ]:
from modules.quality_evaluator import run_quality_report, run_ml_utility_test

quality_results = run_quality_report(full_df, dp_synthetic_df)

In [ ]:
# Show quality scores bar chart
from IPython.display import Image, display

if os.path.exists(config.QUALITY_SCORES_PNG):
    display(Image(config.QUALITY_SCORES_PNG, width=750))

In [ ]:
# ML utility test
ml_results = run_ml_utility_test(full_df, dp_synthetic_df)

print(f'\nTSTR Accuracy (Train-on-Synthetic, Test-on-Real): {ml_results["tstr_accuracy"]*100:.2f}%')
print(f'TRTR Accuracy (Train-on-Real,      Test-on-Real): {ml_results["trtr_accuracy"]*100:.2f}%')
print(f'Utility Gap                                      : {ml_results["utility_gap"]*100:.2f}%')

---
## Step 7: Bias Audit

In [ ]:
from modules.bias_auditor import run_bias_audit

bias_results = run_bias_audit(full_df, dp_synthetic_df)

In [ ]:
# Show bias comparison chart inline
if os.path.exists(config.BIAS_COMPARISON_PNG):
    display(Image(config.BIAS_COMPARISON_PNG, width=750))

print(f'Real data bias score      : {bias_results["real_bias_score"]}')
print(f'Synthetic data bias score : {bias_results["syn_bias_score"]}')
print(f'Bias Reduction            : {bias_results["bias_reduction_pct"]}%')

---
## Conclusion

The following table summarises the results of the full Synthetic Data Generation pipeline:

In [ ]:
import pandas as pd

summary_data = {
    'Metric': [
        'Overall Quality Score',
        'Column Shapes Score',
        'Column Pair Trends Score',
        'Privacy Risk Score (lower=better)',
        'TSTR Accuracy (synthetic → real)',
        'TRTR Accuracy (baseline)',
        'Utility Gap',
        'Bias Score — Real Data',
        'Bias Score — Synthetic Data',
        'Bias Reduction',
    ],
    'Value': [
        f"{quality_results['overall']*100:.2f}%",
        f"{quality_results['shapes']*100:.2f}%",
        f"{quality_results['trends']*100:.2f}%",
        f"{privacy_risk_score:.2f} / 100",
        f"{ml_results['tstr_accuracy']*100:.2f}%",
        f"{ml_results['trtr_accuracy']*100:.2f}%",
        f"{ml_results['utility_gap']*100:.2f}%",
        f"{bias_results['real_bias_score']:.4f}",
        f"{bias_results['syn_bias_score']:.4f}",
        f"{bias_results['bias_reduction_pct']:.2f}%",
    ],
    'Interpretation': [
        'Higher = better statistical fidelity',
        'Distribution match per column',
        'Correlation structure preservation',
        '< 30 = safe; > 70 = at risk',
        'ML usefulness of synthetic data',
        'Oracle (real data) baseline',
        'Small gap = highly useful',
        'Mean DI deviation (0 = fair)',
        'Mean DI deviation (0 = fair)',
        'Positive = fairness improved',
    ],
}

summary_df = pd.DataFrame(summary_data)
summary_df

---

### Key Takeaways

1. **Fidelity**: CTGAN reproduces the statistical distributions of the original Synthea data with high accuracy.
2. **Privacy**: Differential privacy (Laplace, ε=1.0) provides rigorous mathematical guarantees; membership inference risk is low.
3. **Utility**: TSTR accuracy closely tracks TRTR — synthetic data is a viable substitute for ML training.
4. **Fairness**: The synthetic generation process reduces demographic bias compared to the Synthea baseline.

> **All charts are saved to `results/charts/` and all synthetic data to `data/synthetic/synthetic_patients.csv`.**